In [ ]:
#This needs to be added in the main file just below where we get filtered_ds
#save_path = "clinical_trial_simplification_filtered"
#filtered_ds.save_to_disk(save_path)

#print(f"Saved filtered dataset to: {save_path}")

Load Groq Client

In [11]:
!pip install groq
import os
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

   ---------------------------------------- 0.0/137.5 kB ? eta -:--:--
   -- ------------------------------------- 10.2/137.5 kB ? eta -:--:--
   -------------------------------------- - 133.1/137.5 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 137.5/137.5 kB 2.0 MB/s eta 0:00:00


Load Filtered Dataset

In [13]:
from datasets import load_from_disk

ds = load_from_disk("clinical_trial_simplification_filtered")
print("Rows loaded:", len(ds))


Rows loaded: 344475


Function combining the Three Fields for Summarization

In [15]:
def build_input(example):
    desc = example.get("detailed_description") or ""
    brief = example.get("brief_summary") or ""
    elig = example.get("eligibility_criteria") or ""

    return (
        "Detailed Description:\n" + desc.strip() + "\n\n" +
        "Brief Summary:\n" + brief.strip() + "\n\n" +
        "Eligibility Criteria:\n" + elig.strip()
    )


Choose the model

In [22]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

models = client.models.list()

for m in models.data:
    print(m.id)


llama-3.1-8b-instant
moonshotai/kimi-k2-instruct-0905
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
playai-tts-arabic
moonshotai/kimi-k2-instruct
qwen/qwen3-32b
openai/gpt-oss-20b
groq/compound-mini
openai/gpt-oss-safeguard-20b
meta-llama/llama-4-maverick-17b-128e-instruct
playai-tts
groq/compound
allam-2-7b
llama-3.3-70b-versatile
meta-llama/llama-guard-4-12b
meta-llama/llama-4-scout-17b-16e-instruct
whisper-large-v3


Apply Summarization to the Dataset

In [30]:
def summarize_clinical(text):
    prompt = f"""
    Summarize the following clinical trial information into 2–3 clear,
    medically accurate, patient-friendly sentences. Avoid unnecessary jargon.

    {text}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=200,
    )

    return response.choices[0].message.content


Groq Summarization Function which producces 2–3 sentences

In [35]:
from tqdm import tqdm

summaries = []

print("Generating summaries with Llama-3-70B on Groq...")

for row in tqdm(ds, total=len(ds)):
    combined_text = build_input(row)
    summary = summarize_clinical(combined_text)
    summaries.append(summary)


Generating summaries with Llama-3-70B on Groq...


  0%|                                                                          | 97/344475 [07:18<432:16:24,  4.52s/it]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kc93qs1pekdsbawp8sg4dy8z` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99972, Requested 462. Please try again in 6m14.976s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Adding Summaries and Saving New Dataset

In [ ]:
ds = ds.add_column("combined_summary", summaries)

save_path = "clinical_trial_summaries_llama3_70b_groq"
ds.save_to_disk(save_path)

print("Saved summarized dataset to:", save_path)


The above code takes so much time to run. So we are implementing batch processing below to make it robust and fast.
It implements a scalable, fault-tolerant batch summarization pipeline designed for large-scale clinical text processing. Given the size of the dataset (~344,000 records), issuing individual API requests for each entry was computationally prohibitive and susceptible to rate limitations, network faults, and system interruptions. 

To address these challenges, this implementation employs a batch-oriented approach, grouping multiple clinical trial records into a single request to the Groq Llama-3.3-70B-versatile model. The pipeline incorporates several robustness features commonly used in large-scale NLP workflows:
    • Batch processing to reduce the number of API calls and improve throughput  
    • Exponential backoff and retry logic to manage transient API or network failures  
    • Persistent checkpointing of progress, enabling seamless resumption after interruptions without recomputation  
    • Periodic intermediate dataset serialization to safeguard against data loss  
    • Structured JSON-based responses to ensure deterministic parsing and mapping between input trials and generated summaries  

This design supports high-volume text summarization with strong guarantees of reliability and reproducibility. The final output augments the dataset with a "combined_summary" field containing concise, patient-readable (2–3 sentence) abstractive summaries synthesized from the detailed description, brief summary, and eligibility criteria of each clini

Now we have run the code with subset because of the rate limit constraint of groq.cal trial.
f each clinical trial.


In [50]:
from datasets import load_from_disk

# Loading full dataset & create subset
full_ds = load_from_disk("clinical_trial_simplification_filtered")

subset_size = 100
ds = full_ds.select(range(subset_size))

print("Using subset of size:", len(ds))

# 2. BEGIN BATCH PIPELINE
import os
import json
import time
import re
from groq import Groq
from tqdm import tqdm

# Configuration 
MODEL_ID = "llama-3.1-8b-instant"   # much more efficient for testing
BATCH_SIZE = 3
MAX_TOKENS = 300
TEMPERATURE = 0.2
SAVE_EVERY_N_BATCHES = 100       
CHECKPOINT_PATH = "groq_progress_subset.json"
OUTPUT_DS_PATH = "clinical_trial_summaries_subset"  
N = len(ds)

# Safety: load API key from env
if "GROQ_API_KEY" not in os.environ:
    raise RuntimeError("Set your Groq API key in the environment variable GROQ_API_KEY before running this cell.")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print(f"Loaded subset with {N} rows")


# -----------------------------
# Helper: combine fields
# -----------------------------
def build_input_text(example):
    desc = example.get("detailed_description") or ""
    brief = example.get("brief_summary") or ""
    elig = example.get("eligibility_criteria") or ""
    combined = (
        "Detailed Description:\n" + desc.strip() + "\n\n" +
        "Brief Summary:\n" + brief.strip() + "\n\n" +
        "Eligibility Criteria:\n" + elig.strip()
    )
    return combined

# -----------------------------
# Load / Save checkpoint
# -----------------------------
def load_checkpoint(path):
    if os.path.exists(path):
        print("Loading checkpoint:", path)
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data.get("next_index", 0), data.get("summaries", [])
    return 0, []

def save_checkpoint(path, next_index, summaries):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump({"next_index": next_index, "summaries": summaries}, f)
    os.replace(tmp, path)
    print(f"Checkpoint saved. next_index={next_index}, summaries={len(summaries)}")

# -----------------------------
# Prompt builder for JSON output
# -----------------------------
def make_batch_prompt(batch_items):
    parts = []
    parts.append("You are a medical summarization assistant.")
    parts.append("For each trial below, produce a 3-4 sentence, patient-friendly summary. Avoid jargon and be medically accurate.")
    parts.append("Return the summaries as a STRICT JSON array of objects, each containing:")
    parts.append("  - \"index\": the input index")
    parts.append("  - \"summary\": the generated summary\n")
    parts.append('Example: [{"index": 0, "summary": "This trial evaluates ..."}]\n')
    parts.append("Now summarize the following trials:\n")

    for idx, text in batch_items:
        parts.append(f"TRIAL_INDEX: {idx}\n{text}\n")
    return "\n".join(parts)

# -----------------------------
# JSON extraction helper
# -----------------------------
def extract_json_array_from_text(text):
    text = text.strip()
    try:
        return json.loads(text)
    except:
        pass

    m = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if m:
        candidate = m.group(0)
        try:
            return json.loads(candidate)
        except:
            pass
    return None

# -----------------------------
# Batch summarization with retries
# -----------------------------
def summarize_batch(batch_items, max_retries=5, initial_backoff=1.0):
    prompt = make_batch_prompt(batch_items)
    attempt = 0

    while attempt < max_retries:
        attempt += 1
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            raw = response.choices[0].message.content
            parsed = extract_json_array_from_text(raw)

            if parsed is None:
                raise ValueError("Could not parse JSON array.")

            for obj in parsed:
                if "index" not in obj or "summary" not in obj:
                    raise ValueError("JSON missing required keys.")

            return parsed

        except Exception as e:
            print(f"[Warning] Batch attempt {attempt}/{max_retries} failed: {e}")
            if attempt >= max_retries:
                raise
            sleep_time = initial_backoff * (2 ** (attempt - 1))
            print(f"Sleeping {sleep_time:.1f} seconds...")
            time.sleep(sleep_time)

# -----------------------------
# Main summarization loop
# -----------------------------
start_index, completed_summaries = load_checkpoint(CHECKPOINT_PATH)
summary_map = {item["index"]: item["summary"] for item in completed_summaries}

print("Resuming from index:", start_index)

batch_count = 0
i = start_index
pbar = tqdm(total=N, initial=start_index)

try:
    while i < N:
        batch_items = []
        for j in range(i, min(i + BATCH_SIZE, N)):
            if j not in summary_map:
                batch_items.append((j, build_input_text(ds[j])))

        if not batch_items:
            i += BATCH_SIZE
            pbar.update(BATCH_SIZE)
            continue

        parsed = summarize_batch(batch_items)
        for obj in parsed:
            summary_map[obj["index"]] = obj["summary"].strip()

        i_next = min(i + BATCH_SIZE, N)
        batch_count += 1

        save_checkpoint(CHECKPOINT_PATH, i_next, [{"index": k, "summary": v} for k, v in summary_map.items()])

        pbar.update(len(batch_items))
        i = i_next

finally:
    pbar.close()

# -----------------------------
# Save final subset output
# -----------------------------
final_summaries = [summary_map.get(idx, "") for idx in range(N)]
ds = ds.add_column("combined_summary", final_summaries)
ds.save_to_disk(OUTPUT_DS_PATH)

print("Subset summarization complete!")
print("Saved to:", OUTPUT_DS_PATH)

Using subset of size: 100
Loaded subset with 100 rows
Resuming from index: 0


  3%|██▍                                                                               | 3/100 [00:00<00:23,  4.13it/s]

Checkpoint saved. next_index=3, summaries=3


  6%|████▉                                                                             | 6/100 [00:01<00:20,  4.51it/s]

Checkpoint saved. next_index=6, summaries=6
[Warning] Batch attempt 1/5 failed: Could not parse JSON array.
Sleeping 1.0 seconds...


...

Checkpoint saved. next_index=99, summaries=99


100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [19:56<00:00, 11.96s/it]

Checkpoint saved. next_index=100, summaries=100


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Subset summarization complete!
Saved to: clinical_trial_summaries_subset


In [54]:
#see the results 
from datasets import load_from_disk

ds_out = load_from_disk("clinical_trial_summaries_subset")

ds_out.column_names

print(ds_out[0]["combined_summary"])

for i in range(5):
    print(f"--- Summary {i} ---")
    print(ds_out[i]["combined_summary"])
    print()


This trial investigates the effects of the iron-chelating agent deferasirox on patients with iron overload caused by blood transfusions. The goal is to see if deferasirox can safely and effectively remove excess iron from the body.
--- Summary 0 ---
This trial investigates the effects of the iron-chelating agent deferasirox on patients with iron overload caused by blood transfusions. The goal is to see if deferasirox can safely and effectively remove excess iron from the body.

--- Summary 1 ---
This study compares two types of hip replacement implants to see which one lasts longer. Patients will receive either a Taperloc Complete Reduced Distal or a Taperloc Complete Microplasty hip stem, and their progress will be tracked over two years.

--- Summary 2 ---
This trial tests the combination of two drugs, sorafenib and paclitaxel, to treat metastatic breast cancer. The goal is to see if this combination can slow the growth of tumors and improve patient outcomes.

--- Summary 3 ---
This 

**Final Summarization Output**

In [65]:
from datasets import load_from_disk

ds_out = load_from_disk("clinical_trial_summaries_subset")

def show_example(index):
    print("======================================")
    print(f"Example {index}")
    print("======================================\n")

    print("Detailed Description:\n")
    print(ds_out[index]["detailed_description"][:600], "\n")

    print("Brief Summary:\n")
    print(ds_out[index]["brief_summary"][:400], "\n")

    print("Eligibility Criteria:\n")
    print(ds_out[index]["eligibility_criteria"][:400], "\n")

    print("Generated Summary:\n")
    print(ds_out[index]["combined_summary"])
    print("\n--------------------------------------\n")

# Show example 0
show_example(0)


Example 0

📝 Detailed Description:

PRIMARY OBJECTIVES: I. To determine the effects of the iron-chelating agent deferasirox on changes in: neutrophil function; macrophage function; lymphocyte function.

SECONDARY OBJECTIVES: I. To determine the effect of chelation on the incidence of bacterial, viral and fungal infections documented by clinical, microbiologically-proven versus radiologically-proven criteria. II. To determine the effect of iron chelation on mortality and morbidity with incidence of the following parameters: Need for hospitalization; Duration of hospitalization; Need for ventilatory support; Need for exchange tran 

Brief Summary:

RATIONALE: Deferasirox may remove excess iron from the body caused by blood transfusions.

PURPOSE: This clinical trial studies deferasirox in treating iron overload caused by blood transfusions in patients with hematologic malignancies. 

Eligibility Criteria:

Inclusion Criteria:

* Patients must have a pathology confirmed diagnosis of one o

**With flan-t5-small**

In [70]:
!pip install transformers datasets sentencepiece rouge-score nltk


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.1 MB 871.5 kB/s eta 0:00:02
   ---------------- ----------------------- 0.4/1.1 MB 4.6 MB/s eta 0:00:01
   ----------------------------------- ---- 0.9/1.1 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/135.8 kB ? eta -:--:--
   ---------------------------------------- 135.8/135.8 kB 7.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24972 sha256=8cd2cb9c916650aed314ab6025553d1a52b80f0e8d058e9c5330d4220069ff23
  Stored in directory: c:\users\sayan\appdata\local\pip\cache\wheels\85\9d\af\01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [3]:
#load dataset
from datasets import load_from_disk

full_ds = load_from_disk("clinical_trial_simplification_filtered")

# Use 2000 for training on CPU; adjust if needed
subset_size = 2000
ds = full_ds.select(range(subset_size))

print("Loaded dataset with", len(ds), "rows")


Loaded dataset with 2000 rows


We apply light, meaning-preserving preprocessing before training. Each text field (detailed description, brief summary, eligibility criteria) is normalized for Unicode, HTML tags are removed, and whitespace is collapsed. We do not remove stopwords because transformer-based seq2seq models rely on the full syntactic structure to understand negation and eligibility phrasing. To respect model context limits, we truncate inputs at sentence boundaries where possible.

In [6]:

# Preprocessing for T5
# Transformer seq2seq models (T5/FLAN-T5) benefit from clean, meaning-preserving inputs. 
# Doing light normalization: remove HTML, collapse whitespace, fix newlines, normalize unicode. 
# And ,not removing stopwords: removing them can break syntax and change meaning

import re
import unicodedata

def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def safe_truncate(text: str, max_chars: int = 4000) -> str:
    if len(text) <= max_chars:
        return text
    # try to cut at last sentence end
    chunk = text[:max_chars]
    idx = chunk.rfind(". ")
    if idx != -1:
        return chunk[:idx+1]
    return chunk  # fallback

def build_input(example):
    desc = clean_text(example.get("detailed_description", ""))
    brief = clean_text(example.get("brief_summary", ""))
    elig = clean_text(example.get("eligibility_criteria", ""))

    combined = f"""
    Summarize the following clinical trial in 3–4 patient-friendly sentences.

    Detailed Description:
    {desc}

    Brief Summary:
    {brief}

    Eligibility Criteria:
    {elig}
    """

    combined = safe_truncate(combined, 4000)

    example["input_text"] = combined
    example["target_text"] = brief  # model learns from original brief summary
    return example

ds = ds.map(build_input)
print("Preprocessing complete!")



Preprocessing complete!


Using google/flan-t5-small for faster processing

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Using device:", device)


Using device: cpu


In [11]:
#tokenization
max_input_len = 512
max_target_len = 128

def tokenize_batch(batch):
    inputs = tokenizer(
        batch["input_text"],
        padding="max_length",
        truncation=True,
        max_length=max_input_len,
    )
    targets = tokenizer(
        batch["target_text"],
        padding="max_length",
        truncation=True,
        max_length=max_target_len,
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_ds = ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=ds.column_names
)

tokenized_ds = tokenized_ds.train_test_split(test_size=0.1)
train_ds = tokenized_ds["train"]
val_ds = tokenized_ds["test"]

print(train_ds, val_ds)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1800
}) Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


In [113]:
!pip install "transformers[torch]"

  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/380.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/380.9 kB ? eta -:--:--
   ------------ --------------------------- 122.9/380.9 kB 1.8 MB/s eta 0:00:01
   ---------------------------------------  378.9/380.9 kB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 380.9/380.9 kB 3.4 MB/s eta 0:00:00
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.12
    Uninstalling sympy-1.12:
      Successfully uninstalled sympy-1.12


In [17]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_small_ct_model",
    save_strategy="epoch",        # we still save each epoch
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    predict_with_generate=True,
    fp16=False,
)


In [ ]:
#trainer and train
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    data_collator=data_collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
trainer.train()


Step,Training Loss


In [ ]:
#Evaluate the model (using ROUGE)
from evaluate import load
rouge = load("rouge")

def generate_summary_eval(example):
    outputs = model.generate(
        tokenizer(
            example["input_text"],
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).input_ids
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# generate predictions for validation set
preds = []
refs = []

for i in range(100):  # evaluate first 100 examples
    pred = generate_summary_eval(val_ds[i])
    ref = val_ds[i]["cleaned_target"]
    preds.append(pred)
    refs.append(ref)

results = rouge.compute(predictions=preds, references=refs)
print(results)



In [ ]:
#Inference Function
def summarize_trial(text):
    cleaned = clean_text(text)
    prompt = "Summarize the following clinical trial:\n\n" + cleaned

    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512).to(device)

    output = model.generate(
        **inputs,
        max_length=150,
        num_beams=4
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test inference
print(summarize_trial(ds[0]["input_text"]))


In [ ]:
#save model
model.save_pretrained("flan_small_finetuned")
tokenizer.save_pretrained("flan_small_finetuned")
print("Model saved.")
